# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Albert Eli Kwasi Andzi-Quainoo]
**Student ID:** [47762028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [27]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---

from dotenv import load_dotenv
load_dotenv() # Loads .env file into the system environment


# TODO: set API_KEY using ONE of the methods above.

# Using Groq
from groq import Groq

client = Groq(
    api_key = os.environ.get("GROQ_API_KEY"),
)

MODEL = "openai/gpt-oss-20b" # Using Llama 3.1 instant
print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab: - DONE
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant who provides concise and coherent answers. Avoid using filler words.",
             temperature=0.7, max_tokens=500, response_format=None):

          # Request arguments
        request_args = {
                "model": MODEL,
                "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                ],
                "temperature": temperature,
                "max_tokens": max_tokens,
        }
        # Forwarding response_format (for prose to JSON)
        if response_format is not None:
                request_args["response_format"] = response_format

        response = client.chat.completions.create(**request_args)

        return response
#
# TODO: Call it once with a simple question and print the answer. - DONE
answer = ask_llm("What is drafting in racing in one sentence?")
print(answer.choices[0].message.content)



# TODO: Print response.usage as well — how many tokens did your call consume? - 187 tokens (DONE)
print(answer.usage)



Drafting is the technique of following closely behind another racer to reduce air resistance and increase speed.
CompletionUsage(completion_tokens=87, prompt_tokens=100, total_tokens=187, completion_time=0.094284148, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=59), prompt_time=0.005606992, prompt_tokens_details=None, queue_time=0.214935476, total_time=0.09989114)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [Double-click to edit]

### Part 1.2 — Temperature: the randomness dial

In [30]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

prompt = "Suggest a name for a savings product for market traders in Accra."

print("===== Temperature at 0.0=====")
for interaction in range(5):
    answer = ask_llm(prompt, temperature=0.0)
    answer = answer.choices[0].message.content
    print(f"Iteration: {interaction+1}: {answer}")
    print()

print()

print("===== Temperature at 1.2=====")
for interaction in range(5):
    for interaction in range(5):
        answer = ask_llm(prompt, temperature=1.2)
        answer = answer.choices[0].message.content
        print(f"Iteration: {interaction+1}: {answer}")
        print()

print()

===== Temperature at 0.0=====
Iteration: 1: **Accra TradeNest** – a savings product that lets market traders in Accra grow and protect their earnings, with a name that blends local pride and the idea of a secure, growing nest.

Iteration: 2: **Accra TradeNest** – a savings product that lets market traders in Accra grow and protect their earnings, with a name that blends local pride and the idea of a secure, growing nest.

Iteration: 3: **Accra TradeNest** – a savings product that lets market traders in Accra grow and protect their earnings, with a name that blends local pride and the idea of a secure, growing nest.

Iteration: 4: **Accra TradeNest** – a savings product that lets market traders in Accra grow and protect their earnings, with a name that blends local pride and the idea of a secure, growing nest.

Iteration: 5: **Accra TradeNest** – a savings product that lets market traders in Accra grow and protect their earnings, with a name that blends local pride and the idea of a sec

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [Double-click to edit]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [9]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:"). - DONE
#   Run it on L002 and L006. Read the output critically.

# L002 and L006
letters_l002_l006 = ["L002", "L006"]

formatted_letters = ""

for number in letters_l002_l006:
    if number in LETTERS: # Preventing missing key error
        content = LETTERS[number]
        formatted_letters += f"<letter id='{number}'>\n{content}\n</letter>\n\n"


l002_l006_llm_v1_answer = ask_llm(f"Summarize this:\n\n{formatted_letters}\n\n")
print("======== V1 ========")
print(l002_l006_llm_v1_answer.choices[0].message.content)
print()

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with: - DONE
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

l002_l006_llm_v2_answer = ask_llm(system_prompt="You are an assistant to a microfinance loan officer. " \
"You must provide factual and neutral responses with no invented details, in 3-4 sentences.", 
user_prompt= f"Summarize these loan applications:\n\n{formatted_letters}\n\n",
    temperature=0)
print("======== V2 ========")
print(l002_l006_llm_v2_answer.choices[0].message.content)


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


======== V1 ========
Two individuals are writing to request financial assistance:

1. Kwame Boateng, a commercial driver in Kumasi, needs GHS 25,000 to repair his trotro engine and settle personal debts. He promises to repay when his business picks up after the festive season, but lacks collateral.

2. Kofi, a 22-year-old, wants GHS 50,000 to start a car washing business, provision shop, and import phones from Dubai. He claims to be business-minded and trustworthy, but has no collateral. He promises to repay within a year once his businesses boom.

======== V2 ========
Two loan applications have been submitted. 

Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his trotro engine and settle personal debts. He expects business to pick up after the festive season and is willing to repay the loan when funds become available. However, he does not have collateral to secure the loan.

Kofi is applying for GHS 50,000 to start multiple businesses, including a car

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [Double-click to edit]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [48]:
import json
import pandas as pd
from groq import BadRequestError

# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON - DONE
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0



# Obtaining the inner keys
FIELD_NAMES = list(next(iter(GOLD.values())).keys())

FIELD_TYPES = {
    "applicant_name" : {"type": "string"},
    "amount_ghs": {"type": "number"},
    "purpose": {"type": "string"},
    "monthly_profit_ghs": {"type": ["number", "null"]},
    "has_collateral_or_guarantor": {"type": "boolean"},
    "repayment_months" : {"type": ["number", "null"]},
}

# Building the loan schema
LOAN_SCHEMA = {
    "type":"object",
    "properties": {
        field_name:
        FIELD_TYPES[field_name]
        for field_name in FIELD_NAMES
    },
    "required": FIELD_NAMES,
    "additionalProperties" : False,
}

# Wrapping as a response format
RESPONSE_FORMAT = {
    "type":"json_schema",
    "json_schema": {
        "name": "loan_application_extraction",
        "strict": True,
        "schema": LOAN_SCHEMA
    }
}


# Extraction prompt
EXTRACT_PROMPT = """
Extract structured information from the loan application below.

Return ONLY one valid JSON object. Do not include explanations, commentary, or Markdown
code fences.

The JSON object must contain exactly these keys:

- applicant_name: string
- amount_ghs: number
- purpose: string
- monthly_profit_ghs: number or null
- has_collateral_or_guarantor: boolean
- repayment_months: number or null

Extraction rules:
1. Use only information explicitly stated in the letter.
2. If a field is not stated in the letter, use null. Do not guess. 
3. Do not guess or invent missing information.
4. Write monetary values as numbers without "GHS" or commas.
5. Set has_collateral_or_guarantor to true when collateral or a guarantor is explicitly
   offered. Set it to false when the applicant explicitly says neither is available.
6. Convert repayment periods expressed in years into months. For example, one year is 12.
7. Do not add any keys beyond the six listed above.

Worked example:

Letter:
My name is Ama Asare. I need GHS 6,500 to purchase a new oven for my bakery.
My mother has agreed to guarantee the loan. I intend to repay it over 10 months.

JSON:
{{
  "applicant_name": "Ama Asare",
  "amount_ghs": 6500,
  "purpose": "purchase a new oven for her bakery",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}}

Now extract information from this application:

{letter_text}
"""



# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences, - DONE 
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text) -> dict:
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
    try:
        response = ask_llm(
            user_prompt=prompt,
            system_prompt=(
                "You are a precise data-extraction assistant. "
                "Use only facts explicitly stated in the supplied letter."
            ),
            temperature=0,
            response_format=RESPONSE_FORMAT,
        )
    except BadRequestError as error:
         print(f"Warning: API could not produce schema-valid JSON: {error}")
         return None

    raw_output = response.choices[0].message.content.strip() # LLM Response

    # Remove possible Markdown fences
    cleaned_output = (
        raw_output
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    
    # JSON parsing 
    try:
        return json.loads(cleaned_output)
    except json.JSONDecodeError as error:
        print(f"Warning: could not parse the LLM response: {error}")
        return None



# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it. - DONE

rows = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is None:
        extracted = {key: None for key in FIELD_NAMES}

    rows.append({
        "letter_id": letter_id,
        **extracted,
    })

extractions_df = pd.DataFrame(
    rows,
    columns=["letter_id", *FIELD_NAMES],
)

display(extractions_df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,poultry farm at Nsawam,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [52]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# Creating the brief prompt

BRIEF_PROMPT = """
You are assisting a human microfinance loan officer.

Using only the application letter and extracted data provided below, produce a concise
decision-support brief.

Your response must contain exactly these sections:

1. Strengths
- List positive factors explicitly supported by the application.

2. Risks / red flags
- List concerns, inconsistencies, uncertainty, or repayment risks supported by the
  application.

3. Missing information to request
- List information or documents that the applicant did not provide and that the loan
  officer should request.

4. Suggested next step
- Recommend one appropriate procedural action, such as:
  "invite for interview", "request supporting documents", or "flag for senior review".

Rules:
- Use only information contained in the letter or extracted data.
- Do not invent financial information.
- Distinguish missing information from confirmed negative information.
- Do not recommend "approve" or "reject".
- Clearly state that the final lending decision must be made by a human loan officer.
- Keep each section concise.

Treat unsupported applicant statements, predictions, and self-descriptions as
unverified claims, not strengths. Do not describe income as stable or regular unless
the letter provides evidence of consistency. Put absent documents or details only
under Missing information, not under Risks / red flags.

APPLICATION LETTER:
{letter_text}

EXTRACTED DATA:
{extracted_json}
"""

# Helper function to generate the brief

def generate_brief(letter_text, extracted_data):
    prompt = BRIEF_PROMPT.format(
        letter_text = letter_text,
        extracted_json = json.dumps(extracted_data, indent=2),
    )

    response = ask_llm(
        user_prompt=prompt,
        system_prompt=("You provide evidence-based decision support to a human loan officer. "
            "You do not make final lending decisions."),
        temperature=0,
        max_tokens=2000,
    )

    return response.choices[0].message.content.strip()

# Recovering each extraction from rows
extracted_by_id = {
    row["letter_id"]: {
        field_name: row[field_name]
        for field_name in FIELD_NAMES
    }
    for row in rows
}


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

# Generating the briefs for all six letters
briefs = {}

for letter_id, letter_text in LETTERS.items():
    briefs[letter_id] = generate_brief(
        letter_text=letter_text,
        extracted_data=extracted_by_id[letter_id],
    )

# Printing briefs for L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print(f"\n{'=' * 15} {letter_id} {'=' * 15}")
    print(briefs[letter_id])



=============== L001 ===============
**Strengths**  
- 12 years of experience selling provisions at Makola Market.  
- Current stall generates a monthly profit of GHS 900.  
- Has saved GHS 2,500 through the susu scheme and has never missed a contribution.  
- Provides a guarantor (sister, a teacher).  
- Proposes a clear repayment schedule: GHS 450 per month for 20 months.

**Risks / red flags**  
- Monthly profit of GHS 900 leaves only GHS 450 for all other living expenses after the proposed repayment, leaving limited buffer for unforeseen costs.  
- No collateral is offered beyond the guarantor.  
- The application lacks a detailed business plan or evidence that the deep freezer will increase revenue.  
- No documentation of past financial performance or proof of the stated profit.

**Missing information to request**  
- Detailed business plan outlining expected sales and profit after expansion.  
- Recent financial statements or receipts confirming the GHS 900 monthly profit.  
- 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.